In [ ]:
import sys, subprocess, importlib.util, shutil, os, zipfile, io, base64, contextlib
from pathlib import Path

with open(os.devnull, "w") as _null, contextlib.redirect_stdout(_null), contextlib.redirect_stderr(_null):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests", "psutil", "numpy", "ipywidgets"])
if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:
    with open(os.devnull, "w") as _null, contextlib.redirect_stdout(_null), contextlib.redirect_stderr(_null):
        subprocess.run(["bash", "-lc", "apt-get update -qq && apt-get install -y -qq ffmpeg"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

PROJECT_ROOT = Path("/content/kagevietsub-tts")
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
_BUNDLE_B64 = """..."""
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_BUNDLE_B64))) as _zf:
    _zf.extractall(PROJECT_ROOT)

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "capcut-tts-api"))
os.environ.setdefault("PYTHONIOENCODING", "utf-8")


import html, time, traceback
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML, FileLink
from app.core.srt_parser import parse_srt
from app.core.tts_manager import TTSManager
from app.core.state_manager import StateManager
from app.core.retry_manager import RetryManager
from app.core.worker_manager import WorkerManager
from app.audio.timeline import TimelineEngine
from app.audio.mixer import AudioMixer
from app.audio.validator import validate_final_output
from app.utils.paths import sanitize_filename

try:
    from google.colab import output as colab_output, files as colab_files
    colab_output.enable_custom_widget_manager()
except Exception:
    colab_files = None

_voice_manager = TTSManager()
_all_voices = _voice_manager.get_all_voices()
_voice_options = [(f"{v.display_name} · {v.lang or v.lan} · {v.voice_type}", v.voice_type) for v in _all_voices]
_default_voice = _voice_manager.get_default_voice()

_uploaded_srt = {'name': None, 'content': None}
srt_upload = widgets.Button(description='📂 CHỌN FILE SRT', button_style='', layout=widgets.Layout(width='100%', height='42px'))
voice_select = widgets.Dropdown(
    options=_voice_options, value=_default_voice,
    description='Giọng đọc:', style={'description_width':'initial'},
    layout=widgets.Layout(width='100%')
)
threads_select = widgets.IntSlider(
    value=4, min=1, max=30, step=1,
    description='Số luồng TTS:', continuous_update=False,
    style={'description_width':'initial'}, layout=widgets.Layout(width='100%')
)
speed_select = widgets.Dropdown(
    options=[('0.50x',0.50), ('0.75x',0.75), ('0.80x',0.80), ('0.90x',0.90),
             ('1.00x',1.00), ('1.10x',1.10), ('1.20x',1.20), ('1.25x',1.25),
             ('1.50x',1.50), ('2.00x',2.00)],
    value=1.00, description='Tốc độ:', style={'description_width':'initial'},
    layout=widgets.Layout(width='100%')
)
progress = widgets.IntProgress(value=0, min=0, max=100, description='Tiến độ:',
                               bar_style='info', style={'description_width':'initial'},
                               layout=widgets.Layout(width='100%'))
status = widgets.HTML(value='<b>Trạng thái:</b> Chưa tải file SRT.')
preview = widgets.Output(layout={'border':'1px solid #ddd', 'padding':'8px'})
log_output = widgets.Output(layout={'border':'1px solid #ddd', 'padding':'8px', 'max_height':'180px', 'overflow':'auto'})
run_button = widgets.Button(description='▶ BẮT ĐẦU TẠO TTS', button_style='primary',
                            icon='play', layout=widgets.Layout(width='100%', height='42px'))


def _get_uploaded_file():
    return _uploaded_srt['name'], _uploaded_srt['content']


def _choose_srt(_):
    if colab_files is None:
        status.value = '<b style="color:#b91c1c">Vui lòng mở notebook trong Google Colab để chọn file.</b>'
        return
    try:
        picked = colab_files.upload()
        if not picked:
            status.value = '<b>Chưa chọn file SRT.</b>'
            return
        name, content = next(iter(picked.items()))
        if not name.lower().endswith('.srt'):
            status.value = '<b style="color:#b91c1c">Vui lòng chọn file có đuôi .srt.</b>'
            return
        _uploaded_srt['name'] = Path(name).name
        _uploaded_srt['content'] = bytes(content)
        _on_upload_change()
    except Exception as exc:
        status.value = f'<b style="color:#b91c1c">Không tải được file: {html.escape(str(exc))}</b>'


def _on_upload_change(change=None):
    name, content = _get_uploaded_file()
    if not content:
        return
    temp_path = Path('/content') / Path(name).name
    temp_path.write_bytes(content)
    try:
        items = parse_srt(temp_path)
        with preview:
            clear_output(wait=True)
            if not items:
                display(HTML('<span style="color:#b91c1c">Không đọc được phụ đề hợp lệ. Hãy kiểm tra file SRT.</span>'))
                status.value = '<b>Trạng thái:</b> File SRT không hợp lệ.'
                return
            max_end = max((it.end_ms for it in items), default=0)
            display(HTML(f'<b>{html.escape(Path(name).name)}</b> — {len(items)} dòng phụ đề · thời lượng cuối SRT khoảng {max_end/1000:.1f} giây.'))
            rows = ''.join(
                f'<tr><td style="padding:4px;border-bottom:1px solid #ddd">{it.id}</td>'
                f'<td style="padding:4px;border-bottom:1px solid #ddd;white-space:nowrap">{html.escape(it.start_str)} → {html.escape(it.end_str)}</td>'
                f'<td style="padding:4px;border-bottom:1px solid #ddd">{html.escape(it.text[:180])}</td></tr>'
                for it in items[:5]
            )
            display(HTML('<table style="width:100%;border-collapse:collapse"><thead><tr><th align="left">#</th><th align="left">Thời gian</th><th align="left">Nội dung (tối đa 5 dòng đầu)</th></tr></thead><tbody>'+rows+'</tbody></table>'))
        status.value = f'<b>Trạng thái:</b> Đã nạp {len(items)} dòng từ {html.escape(Path(name).name)}.'
    except Exception as exc:
        status.value = f'<b>Trạng thái:</b> Lỗi đọc SRT: {html.escape(str(exc))}'

srt_upload.on_click(_choose_srt)


def _run_pipeline(_):
    name, content = _get_uploaded_file()
    if not content:
        status.value = '<b style="color:#b91c1c">Hãy tải file SRT trước.</b>'
        return
    run_button.disabled = True
    progress.value = 0
    progress.bar_style = 'info'
    with log_output:
        clear_output(wait=True)
    out_dir = Path('/content/kagevietsub_tts_output')
    out_dir.mkdir(parents=True, exist_ok=True)
    srt_path = Path('/content') / Path(name).name
    srt_path.write_bytes(content)
    job_name = sanitize_filename(srt_path.stem)
    try:
        items = parse_srt(srt_path)
        if not items:
            raise ValueError('Không có dòng phụ đề SRT hợp lệ.')
        selected_voice = voice_select.value
        speed = float(speed_select.value)
        workers = int(threads_select.value)
        for item in items:
            item.voice = selected_voice

        state_mgr = StateManager(job_name)
        state_mgr.init_job(items, selected_voice, speed)
        retry_mgr = RetryManager(state_mgr)
        tts_mgr = TTSManager()
        start = time.time()

        def on_progress(completed, total, stats):
            progress.value = int(100 * completed / total) if total else 0
            status.value = (f"<b>Đang tạo:</b> {completed}/{total} dòng · "
                            f"Thành công {stats.get('completed',0)} · "
                            f"Đang xử lý {stats.get('generating',0)} · "
                            f"Lỗi {stats.get('failed',0)} · "
                            f"Đã chạy {time.time()-start:.0f} giây")

        worker = WorkerManager(
            tts_manager=tts_mgr,
            state_manager=state_mgr,
            retry_manager=retry_mgr,
            num_workers=workers,
            speed=speed,
            progress_callback=on_progress,
        )
        status.value = '<b>Trạng thái:</b> Đang gửi yêu cầu tạo giọng đọc đến CapCut TTS…'
        success = worker.process_items(items, selected_voice)
        done = sum(1 for it in items if it.status == 'Hoàn thành' and it.audio_path)
        if not success or done == 0:
            raise RuntimeError(f'Không tạo thành công dòng nào. Kiểm tra lỗi API/Internet; trạng thái hoàn tất: {done}/{len(items)}.')

        status.value = '<b>Trạng thái:</b> Đang ghép âm thanh theo mốc thời gian SRT…'
        timeline = TimelineEngine(items)
        out_file = out_dir / f'{job_name}_TTS.wav'
        mixer = AudioMixer(timeline)
        mixer.merge(out_file.name, custom_output_dir=out_dir, job_name=job_name,
                    progress_cb=lambda pct: progress.__setattr__('value', int(pct*100)))
        ok, message = validate_final_output(out_file)
        if not ok:
            raise RuntimeError(message)
        progress.value = 100
        progress.bar_style = 'success'
        status.value = (f'<b style="color:#15803d">Hoàn tất!</b> {html.escape(out_file.name)} · '
                        f'{out_file.stat().st_size/(1024*1024):.1f} MB · {html.escape(message)}')
        with log_output:
            print('Tạo xong:', out_file)
            print('Voice:', selected_voice, '| Tốc độ:', f'{speed:.2f}x', '| Luồng:', workers)
            print('Các dòng được tạo:', f'{done}/{len(items)}')
            display(FileLink(str(out_file), result_html_prefix='Tải file WAV: '))
        drive_saved_path = None
        try:
            from google.colab import drive as colab_drive
            status.value = '<b>Trạng thái:</b> Đang kết nối Google Drive để lưu WAV thành phẩm…'
            colab_drive.mount('/content/drive', force_remount=False)
            drive_dir = Path('/content/drive/MyDrive/KAGEVIETSUB_TTS')
            drive_dir.mkdir(parents=True, exist_ok=True)
            drive_saved_path = drive_dir / out_file.name
            shutil.copy2(out_file, drive_saved_path)
            status.value = (f'<b style="color:#15803d">Hoàn tất!</b> Đã lưu WAV vào Google Drive: '
                            f'MyDrive/KAGEVIETSUB_TTS/{html.escape(out_file.name)}')
            with log_output:
                print('Đã lưu bản WAV vào Google Drive:', drive_saved_path)
        except Exception as drive_exc:
            with log_output:
                print('Không lưu được vào Google Drive:', repr(drive_exc))
                print('Bạn vẫn có thể tải WAV trực tiếp bên dưới; hãy kiểm tra quyền truy cập Drive nếu muốn lưu lên Drive.')
        with log_output:
            print('Tạo xong:', out_file)
            print('Voice:', selected_voice, '| Tốc độ:', f'{speed:.2f}x', '| Luồng:', workers)
            print('Các dòng được tạo:', f'{done}/{len(items)}')
            display(FileLink(str(out_file), result_html_prefix='Tải file WAV về thiết bị: '))
        if colab_files is not None:
            colab_files.download(str(out_file))
    except Exception as exc:
        progress.bar_style = 'danger'
        status.value = f'<b style="color:#b91c1c">Có lỗi:</b> {html.escape(str(exc))}'
        with log_output:
            print('LỖI:', repr(exc))
            print('Gợi ý: thử giảm số luồng xuống 2–4, kiểm tra Internet/API và thử lại.')
    finally:
        run_button.disabled = False

run_button.on_click(_run_pipeline)

ui = widgets.VBox([
    widgets.HTML('<b>Chọn phụ đề SRT</b>'), srt_upload, preview,
    widgets.HTML('<b>Thiết lập giọng đọc</b>'), voice_select, threads_select, speed_select,
    run_button, progress, status,
    widgets.HTML('<b>Nhật ký / kết quả</b>'), log_output
], layout=widgets.Layout(width='100%'))
display(ui)
